# 🏆 iRage — Short-Horizon Return Prediction | v4 ONE-SHOT FINAL

```
COMPETITION CONSTRAINT: ONE SUBMISSION ONLY — NO RETRY
```

### This notebook is designed around ONE SUBMISSION = ONE CHANCE.
Every cell has a purpose. Run top-to-bottom. Two guaranteed outputs:
1. **Printed R² scorecard** — your offline estimate of model quality
2. **submission.csv** — the file you upload to Kaggle

### Design decisions (one-submission context):
| Decision | Choice | Why |
|---|---|---|
| Validation | 5-Fold KFold OOF | Best offline estimate of private LB |
| Overfit check | Train vs Val gap printed | You MUST verify before submitting |
| Model depth | Shallow (num_leaves=31, depth=4) | Noise-robust, generalizes to 74% private data |
| Regularization | Strong (λ=5) | Private LB has 74% unseen rows — must generalize |
| Ensemble | Simple average LGB+XGB | Robust — optimizer can overfit to OOF noise |
| Target clip | P2/P98 | Remove microstructure noise from training signal |
| Submission guard | 5 hard assertions before save | Crash loud if anything is wrong |

In [32]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — IMPORTS                                           ║
# ╚══════════════════════════════════════════════════════════════╝
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost ✅')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed — LGB+XGB ensemble only')

SEED = 23
np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
print(f'LightGBM {lgb.__version__} | XGBoost {xgb.__version__}')
print('Imports ✅')

CatBoost ✅
LightGBM 4.6.0 | XGBoost 3.2.0
Imports ✅


In [33]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — CONFIG                                            ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ONE-SUBMISSION CONTEXT: Every hyperparameter is chosen for
# GENERALIZATION to private LB (74% of test data), NOT for
# maximizing public LB (only 26% of test data).
#
# Key principle: underfit slightly > overfit significantly
# In low-signal (R²≈0.008) regime, overfitting costs more
# than underfitting.

class CFG:
    # ── Paths ──────────────────────────────────────────────────
    DATA_DIR   = Path('./input')
    TRAIN_PATH = DATA_DIR / 'train-001.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
    OUTPUT_DIR = Path('./output')

    # ── CV ─────────────────────────────────────────────────────
    N_FOLDS = 5
    SEED    = 23

    # ── Target clipping ────────────────────────────────────────
    # P2/P98: short-horizon returns are fat-tailed (kurtosis >> 3).
    # Top/bottom 2% = microstructure noise, not signal.
    # Training on them teaches the model to chase outliers.
    TARGET_CLIP  = True
    CLIP_LOWER   = 2    # lower percentile
    CLIP_UPPER   = 98   # upper percentile

    # ── Feature Engineering ────────────────────────────────────
    BUILD_MOMENTUM    = True   # Acc12, Acc23, Div13 — velocity signals
    BUILD_RATIO       = False  # DISABLED: unstable on anonymous scale
    BUILD_FAMILY_AGG  = True   # Cross-instrument regime signals
    BUILD_PRICE_FEATS = True   # Price is the target driver

    # ── LightGBM ───────────────────────────────────────────────
    # num_leaves=31:     LGB default. 127 memorizes noise.
    # min_child_samples=200: each leaf = 200+ rows. Forces
    #                    the model to find PATTERNS not anomalies.
    # lambda_l1=1, l2=5: aggressive regularization for weak signal.
    # min_gain_to_split=0.01: ignore trivial splits.
    # feature_fraction=0.5: sub-sample features per tree.
    LGB_PARAMS = {
        'objective':         'regression',
        'metric':            'rmse',
        'learning_rate':     0.02,
        'num_leaves':        31,
        'max_depth':         -1,
        'min_child_samples': 300,       #2 Improvement 200 -> 300
        'feature_fraction':  0.5,
        'bagging_fraction':  0.7,
        'bagging_freq':      1,
        'lambda_l1':         1.0,       #1 No improvement  
        'lambda_l2':         8.0,       #1 Improvement 5.0 -> 8.0
        'min_gain_to_split': 0.01,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }
    LGB_NUM_ROUNDS = 12000    #1 Improvement *2    #3 Improvement *2 -> *3    #4 Improvement *3 -> *4    #4 Improvement 5000*4 -> 12000
    LGB_EARLY_STOP = 150*2  #1 Improvement *2

    # ── XGBoost ────────────────────────────────────────────────
    # max_depth=4: shallower than typical (6). Depth 6 with
    #             600k+ rows can perfectly memorize noise.
    # gamma=0.1:  minimum loss reduction to make a split.
    # min_child_weight=200: matches LGB min_child_samples.
    XGB_PARAMS = {
        'objective':             'reg:squarederror',
        'n_estimators':          5000,
        'learning_rate':         0.02,
        'max_depth':             4,
        'min_child_weight':      200,
        'subsample':             0.7,
        'colsample_bytree':      0.5,
        'reg_alpha':             1.0,
        'reg_lambda':            5.0,
        'gamma':                 0.1,
        'tree_method':           'hist',
        'verbosity':             0,
        'n_jobs':                -1,
        'random_state':          SEED,
        'early_stopping_rounds': 150,
    }

    # ── CatBoost (if available) ────────────────────────────────
    CAT_PARAMS = dict(
        iterations=5000,
        learning_rate=0.02,
        depth=4,
        l2_leaf_reg=10.0,
        min_data_in_leaf=200,
        random_seed=SEED,
        eval_metric='RMSE',
        od_type='Iter',
        od_wait=150,
        verbose=500,
        task_type='CPU',
        allow_writing_files=False,
    )

print('Config ✅')

Config ✅


In [34]:
from pathlib import Path
import os

# Change this to your actual data folder path
data_dir = Path('data')

print(f'Folder exists: {data_dir.exists()}')
print(f'\nFiles found:')
for f in sorted(data_dir.rglob('*')):
    if f.is_file():
        print(f'  {f.name}  →  {f}  ({f.stat().st_size/1e6:.1f} MB)')

# Print where the notebook file itself is located
print("Notebook location:", os.getcwd())

# Check all likely locations
candidates = [
    Path('input'),
    Path('data'),
    Path('../input'),
    Path('.'),
    Path('./input'),
    Path('./data'),
]

for p in candidates:
    if p.exists():
        print(f'\n✅ FOUND: {p}')
        for f in sorted(p.iterdir()):
            print(f'   {f.name}  ({f.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'❌ Missing: {p}')

Folder exists: False

Files found:
Notebook location: <repo>\algo1

✅ FOUND: input
   sample_submission.csv  (4.6 MB)
   test.parquet  (1570.2 MB)
   train-001.parquet  (2459.2 MB)
❌ Missing: data
❌ Missing: ../input

✅ FOUND: .
   .venv  (0.0 MB)
   code_try_quant.py  (0.0 MB)
   input  (0.0 MB)
   irage_final_v4.ipynb  (0.1 MB)
   output  (0.0 MB)
   output1  (0.0 MB)

✅ FOUND: input
   sample_submission.csv  (4.6 MB)
   test.parquet  (1570.2 MB)
   train-001.parquet  (2459.2 MB)
❌ Missing: data


In [35]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA LOADING                                      ║
# ╚══════════════════════════════════════════════════════════════╝
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'NaN — Train: {train.isnull().sum().sum():,} | Test: {test.isnull().sum().sum():,}')

# Stash IDs & target BEFORE any modification
train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

train.drop(columns=['ID', 'TARGET'], inplace=True)
test.drop(columns=['ID'],            inplace=True)

# Columns must already match — if not, something is wrong with parquet
assert list(train.columns) == list(test.columns), \
    f'FATAL: train/test column mismatch! {len(train.columns)} vs {len(test.columns)}'

print(f'Features: {train.shape[1]}')
print(f'Train IDs: {train_ids.min()} – {train_ids.max()}')
print(f'Test IDs:  {test_ids.min()} – {test_ids.max()}')

Loading data...
Train: (661574, 447) | Test: (410139, 446)
Loaded in 8.1s
NaN — Train: 0 | Test: 0
Features: 445
Train IDs: 0 – 672373
Test IDs:  672374 – 1089244


In [36]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — TARGET ANALYSIS & CLIPPING                        ║
# ╚══════════════════════════════════════════════════════════════╝
#
# WHY THIS MATTERS FOR A ONE-SHOT SUBMISSION:
# You can't iterate. So the model MUST be trained on clean signal.
# Financial return distributions have kurtosis >> 3 (fat tails).
# The extreme 2% on each side:
#   - are outlier events (flash crashes, errors, illiquidity)
#   - do NOT generalize to the private test set
#   - hurt R² by pulling model predictions toward extremes
# Clipping is NOT data leakage — we clip train target only.
# Test predictions are NOT clipped (let the model predict freely).

print('══════════════════════════════════════════════')
print('TARGET DISTRIBUTION (raw)')
print('══════════════════════════════════════════════')
desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'\nSkewness:  {y_raw.skew():.4f}  (>0 = right-tailed)')
print(f'Kurtosis:  {y_raw.kurtosis():.4f}  (>3 = fat tails, leptokurtic)')

if CFG.TARGET_CLIP:
    clip_lo = y_raw.quantile(CFG.CLIP_LOWER / 100)
    clip_hi = y_raw.quantile(CFG.CLIP_UPPER / 100)
    y_series  = y_raw.clip(clip_lo, clip_hi)
    n_clipped = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
    pct_clip  = 100 * n_clipped / len(y_raw)
    print(f'\nClip: [{clip_lo:.6f}, {clip_hi:.6f}]')
    print(f'Rows clipped: {n_clipped:,} / {len(y_raw):,} ({pct_clip:.2f}%)')
    print(f'Std before: {y_raw.std():.6f}  →  after: {y_series.std():.6f}')
    y = y_series.values.astype(np.float32)
else:
    y = y_raw.values.astype(np.float32)

print('\nTarget ready ✅')

══════════════════════════════════════════════
TARGET DISTRIBUTION (raw)
══════════════════════════════════════════════
count    661574.000000
mean         -0.000036
std           0.036660
min          -1.285424
1%           -0.097335
2%           -0.075900
5%           -0.051777
25%          -0.016153
50%           0.000000
75%           0.016073
95%           0.051537
98%           0.075723
99%           0.097421
max           1.346982
Name: TARGET, dtype: float64

Skewness:  0.1691  (>0 = right-tailed)
Kurtosis:  48.0903  (>3 = fat tails, leptokurtic)

Clip: [-0.075900, 0.075723]
Rows clipped: 26,464 / 661,574 (4.00%)
Std before: 0.036660  →  after: 0.029513

Target ready ✅


In [37]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FEATURE CATEGORIZATION                            ║
# ╚══════════════════════════════════════════════════════════════╝
all_cols  = train.columns.tolist()

lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    """First prefix segment = feature family (S01, S02, SO3 etc.)"""
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

# Map each base col to its lag versions (existence-checked)
lag_map = {
    c: {
        'lag1': c + '_LagT1' if (c + '_LagT1') in lag_set else None,
        'lag2': c + '_LagT2' if (c + '_LagT2') in lag_set else None,
        'lag3': c + '_LagT3' if (c + '_LagT3') in lag_set else None,
    }
    for c in pure_base_cols
}

print(f'Base (incl. special):  {len(base_cols):4d}')
print(f'Pure base:             {len(pure_base_cols):4d}')
print(f'LagT1 features:        {len(lag1_cols):4d}')
print(f'LagT2 features:        {len(lag2_cols):4d}')
print(f'LagT3 features:        {len(lag3_cols):4d}')
print(f'Families found:        {families}')
print(f'Total original feats:  {len(all_cols):4d}')
print('Categorization ✅')

Base (incl. special):   112
Pure base:              110
LagT1 features:         111
LagT2 features:         111
LagT3 features:         111
Families found:        ['S01', 'S02', 'S03', 'S04', 'S05']
Total original feats:   445
Categorization ✅


In [38]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — FEATURE ENGINEERING                               ║
# ╚══════════════════════════════════════════════════════════════╝
#
# QUANT LOGIC FOR EACH FEATURE:
#
#  Acc12 = LagT1 - LagT2
#    → Is short-term momentum ACCELERATING or decelerating?
#    → Accelerating momentum = likely continuation signal
#
#  Acc23 = LagT2 - LagT3
#    → Is medium-term momentum changing?
#    → Regime-change detector (trend flip early warning)
#
#  Div13 = LagT1 - LagT3
#    → Short vs long-term divergence
#    → Large positive = strong trend; near zero = mean-reversion zone
#
#  Family BaseMean/Std
#    → Mean: what is the average 'level' across instruments in this family?
#    → Std: dispersion = how much disagreement within family?
#    → High std = divergence within family = potential regime stress
#
#  Family MomMean/MomStd
#    → Are all instruments in the family moving TOGETHER? (systemic)
#    → High mean + low std = strong coordinated momentum
#
#  Price_Acc, Price_Div13, Price_Acc23
#    → Same logic applied specifically to Price (the target driver)
#    → Price features get extra treatment because TARGET = f(Price)
#
#  SO3_T_sq
#    → SO3_T is the only domain-identified covariate
#    → Likely time-of-day or session marker (intraday pattern)
#    → Squared captures U-shaped or inverted-U intraday effects
#    → E.g. returns may be highest at open/close (quadratic in time)
#
#  WHAT IS NOT BUILT AND WHY:
#    ❌ NormMom = LagT1/base: anonymous features have unknown scale.
#       Division by near-zero base = extreme values even after clipping.
#       Introduces hard non-linearities that mislead the model.
#    ❌ Pairwise interactions: with 800+ features, products = 300k+
#       features. Curse of dimensionality at R²≈0.008 signal level.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # ── A. MOMENTUM (acceleration, jerk, divergence) ──────────
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for base, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2:
                df[base + '_Acc12'] = df[l1] - df[l2]
                new_feats.append(base + '_Acc12')
            if l2 and l3:
                df[base + '_Acc23'] = df[l2] - df[l3]
                new_feats.append(base + '_Acc23')
            if l1 and l3:
                df[base + '_Div13'] = df[l1] - df[l3]
                new_feats.append(base + '_Div13')
        if verbose:
            print(f'  Momentum (Acc12/Acc23/Div13): {len(new_feats)-n0:5d}')

    # ── B. PRICE-SPECIFIC FEATURES ────────────────────────────
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        p_l1 = 'Price_LagT1'
        p_l2 = 'Price_LagT2'
        p_l3 = 'Price_LagT3'
        if p_l1 in df.columns and p_l2 in df.columns:
            df['Price_Acc']   = df[p_l1] - df[p_l2]
            new_feats.append('Price_Acc')
        if p_l1 in df.columns and p_l3 in df.columns:
            df['Price_Div13'] = df[p_l1] - df[p_l3]
            new_feats.append('Price_Div13')
        if p_l2 in df.columns and p_l3 in df.columns:
            df['Price_Acc23'] = df[p_l2] - df[p_l3]
            new_feats.append('Price_Acc23')
        if verbose:
            print(f'  Price-specific:               {len(new_feats)-n0:5d}')

    # ── C. FAMILY AGGREGATES ──────────────────────────────────
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fam_base = [c for c in pure_base_cols
                        if get_family(c) == fam and c in df.columns]
            fam_lag1 = [c + '_LagT1' for c in fam_base
                        if c + '_LagT1' in df.columns]
            if len(fam_base) < 2:
                continue
            df[fam + '_BaseMean'] = df[fam_base].mean(axis=1)
            df[fam + '_BaseStd']  = df[fam_base].std(axis=1)
            new_feats += [fam + '_BaseMean', fam + '_BaseStd']
            if len(fam_lag1) >= 2:
                df[fam + '_MomMean'] = df[fam_lag1].mean(axis=1)
                df[fam + '_MomStd']  = df[fam_lag1].std(axis=1)
                new_feats += [fam + '_MomMean', fam + '_MomStd']
        if verbose:
            print(f'  Family aggregates:            {len(new_feats)-n0:5d}')

    # ── D. SO3_T (published covariate) ───────────────────────
    if 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T'] ** 2
        new_feats.append('SO3_T_sq')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        if verbose:
            print(f'  SO3_T interactions:           {len(new_feats)-n0:5d}')

    if verbose:
        print(f'  ─────────────────────────────────────')
        print(f'  New features:                 {len(new_feats):5d}')
        print(f'  Total features after FE:      {df.shape[1]:5d}')
    return df, new_feats


print('Running feature engineering...')
t0 = time.time()
train_fe, new_feats = feature_engineering(
    train, lag_map, families, pure_base_cols, verbose=True)
test_fe,  _         = feature_engineering(
    test,  lag_map, families, pure_base_cols, verbose=False)
print(f'\nDone in {time.time()-t0:.1f}s')
print(f'Train_FE: {train_fe.shape} | Test_FE: {test_fe.shape}')

Running feature engineering...
  Momentum (Acc12/Acc23/Div13):   330
  Price-specific:                   3
  Family aggregates:               16
  SO3_T interactions:               2
  ─────────────────────────────────────
  New features:                   351
  Total features after FE:        796

Done in 27.4s
Train_FE: (661574, 796) | Test_FE: (410139, 796)


In [39]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — CLEAN & FINALIZE ARRAYS                           ║
# ╚══════════════════════════════════════════════════════════════╝
# Replace inf/-inf with NaN:
# Lag differences can produce inf if one value is +inf or -inf
# in parquet (rare but possible with float32 overflow).
# LGB and XGB both handle NaN natively — they route NaN down
# the optimal branch during training.
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)

# Strict column alignment check
assert list(train_fe.columns) == list(test_fe.columns), \
    'FATAL: Column mismatch between train_fe and test_fe!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)

print(f'Total features:       {n_features:5d}')
print(f'NaN in train_fe:      {train_fe.isnull().sum().sum():10,}')
print(f'NaN in test_fe:       {test_fe.isnull().sum().sum():10,}')

# Convert to float32 numpy (saves ~50% memory vs float64)
X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)

print(f'\nX:      {X.shape} | dtype: {X.dtype}')
print(f'X_test: {X_test.shape} | dtype: {X_test.dtype}')
print(f'y:      {y.shape} | dtype: {y.dtype}')

del train_fe, test_fe, train, test
gc.collect()
print('Memory freed ✅')

Total features:         796
NaN in train_fe:               0
NaN in test_fe:                0

X:      (661574, 796) | dtype: float32
X_test: (410139, 796) | dtype: float32
y:      (661574,) | dtype: float32
Memory freed ✅


In [40]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — OVERFIT DIAGNOSTIC (ONE-SHOT SAFETY CHECK)        ║
# ╚══════════════════════════════════════════════════════════════╝
#
# CRITICAL FOR ONE-SUBMISSION COMPETITION:
# Before committing to 5-fold training (which takes time),
# do a QUICK single-fold check to verify:
#   1. Train R² >> Val R²? → OVERFIT → increase regularization
#   2. Both R² ≈ 0?        → UNDERFIT or no signal
#   3. Both R² > 0 with small gap? → PROCEED with full CV
#
# Target: train_R² - val_R² < 0.01
# If gap > 0.02: STOP, increase lambda_l1/l2 in CFG, restart.

print('══════════════════════════════════════════════')
print('OVERFIT DIAGNOSTIC (fold 1 of 5, quick check)')
print('══════════════════════════════════════════════')

kf_diag  = KFold(n_splits=5, shuffle=True, random_state=CFG.SEED)
trn_d, val_d = next(iter(kf_diag.split(X, y)))

dtrain_d = lgb.Dataset(X[trn_d], label=y[trn_d],
                        feature_name=feature_cols, free_raw_data=True)
dval_d   = lgb.Dataset(X[val_d], label=y[val_d],
                        reference=dtrain_d, free_raw_data=True)

diag_model = lgb.train(
    CFG.LGB_PARAMS,
    dtrain_d,
    num_boost_round=CFG.LGB_NUM_ROUNDS,
    valid_sets=[dtrain_d, dval_d],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
        lgb.log_evaluation(500),
    ],
)

best_it = diag_model.best_iteration
train_r2_diag = r2_score(y[trn_d], diag_model.predict(X[trn_d], num_iteration=best_it))
val_r2_diag   = r2_score(y[val_d], diag_model.predict(X[val_d], num_iteration=best_it))
gap           = train_r2_diag - val_r2_diag

print(f'\nBest iteration:  {best_it}')
print(f'Train R²:        {train_r2_diag:.6f}')
print(f'Val   R²:        {val_r2_diag:.6f}')
print(f'Gap (train-val): {gap:.6f}', end='  ')

if gap < 0.005:
    print('✅ EXCELLENT — very low overfit, safe to submit')
elif gap < 0.01:
    print('✅ GOOD — acceptable for one-shot submission')
elif gap < 0.02: 
    print('⚠️  MODERATE — consider increasing lambda_l2 before submitting')
else:
    print('❌ HIGH OVERFIT — STOP. Increase lambda_l1/l2 or min_child_samples')
    print('   Do NOT submit with this overfit level in a one-shot competition!')

del diag_model, dtrain_d, dval_d
gc.collect()
print('\nDiagnostic ✅ — proceeding to full 5-fold CV')

══════════════════════════════════════════════
OVERFIT DIAGNOSTIC (fold 1 of 5, quick check)
══════════════════════════════════════════════
[500]	train's rmse: 0.0291047	valid's rmse: 0.0290756
[1000]	train's rmse: 0.0290195	valid's rmse: 0.0290258
[1500]	train's rmse: 0.0289786	valid's rmse: 0.029003
[2000]	train's rmse: 0.0289519	valid's rmse: 0.0289878
[2500]	train's rmse: 0.0289337	valid's rmse: 0.0289782
[3000]	train's rmse: 0.0289173	valid's rmse: 0.0289695
[3500]	train's rmse: 0.0289047	valid's rmse: 0.0289628
[4000]	train's rmse: 0.028895	valid's rmse: 0.0289577
[4500]	train's rmse: 0.0288861	valid's rmse: 0.0289534
[5000]	train's rmse: 0.0288771	valid's rmse: 0.0289483
[5500]	train's rmse: 0.0288707	valid's rmse: 0.028945
[6000]	train's rmse: 0.0288641	valid's rmse: 0.0289416
[6500]	train's rmse: 0.0288586	valid's rmse: 0.0289389
[7000]	train's rmse: 0.0288537	valid's rmse: 0.0289366
[7500]	train's rmse: 0.0288485	valid's rmse: 0.0289339
[8000]	train's rmse: 0.0288434	valid's 

In [41]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — LIGHTGBM: 5-FOLD CV (NATIVE API)                  ║
# ╚══════════════════════════════════════════════════════════════╝
kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

oof_lgb      = np.zeros(len(y), dtype=np.float64)
pred_lgb     = np.zeros(len(X_test), dtype=np.float64)
lgb_scores   = []
lgb_iters    = []
lgb_models   = []

print('══════════════════════════════════════════════')
print(f'LightGBM  —  {CFG.N_FOLDS}-Fold Cross-Validation')
print('══════════════════════════════════════════════')

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    t0 = time.time()
    X_trn, X_val = X[trn_idx], X[val_idx]
    y_trn, y_val = y[trn_idx], y[val_idx]

    dtrain = lgb.Dataset(X_trn, label=y_trn,
                          feature_name=feature_cols, free_raw_data=True)
    dval   = lgb.Dataset(X_val, label=y_val,
                          reference=dtrain, free_raw_data=True)

    model = lgb.train(
        CFG.LGB_PARAMS,
        dtrain,
        num_boost_round=CFG.LGB_NUM_ROUNDS,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(CFG.LGB_EARLY_STOP, verbose=False),
            lgb.log_evaluation(500),
        ],
    )

    best_it = model.best_iteration
    lgb_models.append(model)
    lgb_iters.append(best_it)

    val_preds = model.predict(X_val, num_iteration=best_it)
    oof_lgb[val_idx] = val_preds
    pred_lgb += model.predict(X_test, num_iteration=best_it) / CFG.N_FOLDS

    fold_r2 = r2_score(y_val, val_preds)
    lgb_scores.append(fold_r2)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS}  |  Val R²: {fold_r2:+.6f}  |  '
          f'iter: {best_it}  |  {time.time()-t0:.0f}s')

lgb_oof_r2 = r2_score(y, oof_lgb)
lgb_std    = np.std(lgb_scores)

print(f'\n  LGB OOF R²:   {lgb_oof_r2:+.6f}')
print(f'  Fold std:     {lgb_std:.6f}')
print(f'  Stable?       {"✅ YES" if lgb_std < 0.002 else "⚠️  HIGH VARIANCE"}')
print(f'  Avg best iter:{np.mean(lgb_iters):.0f}')

══════════════════════════════════════════════
LightGBM  —  5-Fold Cross-Validation
══════════════════════════════════════════════
[500]	train's rmse: 0.0291047	valid's rmse: 0.0290756
[1000]	train's rmse: 0.0290195	valid's rmse: 0.0290258
[1500]	train's rmse: 0.0289786	valid's rmse: 0.029003
[2000]	train's rmse: 0.0289519	valid's rmse: 0.0289878
[2500]	train's rmse: 0.0289337	valid's rmse: 0.0289782
[3000]	train's rmse: 0.0289173	valid's rmse: 0.0289695
[3500]	train's rmse: 0.0289047	valid's rmse: 0.0289628
[4000]	train's rmse: 0.028895	valid's rmse: 0.0289577
[4500]	train's rmse: 0.0288861	valid's rmse: 0.0289534
[5000]	train's rmse: 0.0288771	valid's rmse: 0.0289483
[5500]	train's rmse: 0.0288707	valid's rmse: 0.028945
[6000]	train's rmse: 0.0288641	valid's rmse: 0.0289416
[6500]	train's rmse: 0.0288586	valid's rmse: 0.0289389
[7000]	train's rmse: 0.0288537	valid's rmse: 0.0289366
[7500]	train's rmse: 0.0288485	valid's rmse: 0.0289339
[8000]	train's rmse: 0.0288434	valid's rmse: 0.0

In [42]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — FEATURE IMPORTANCE + STABILITY ANALYSIS          ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ONE-SHOT INSIGHT: A feature that appears important in only
# 2 out of 5 folds is probably fitting noise in those folds.
# 'stability' = fraction of folds where importance > 0.
# Good features: stability ≥ 0.8 (important in 4+ folds)
# Noise features: stability < 0.6 (important in <3 folds)

imp_per_fold = np.array([m.feature_importance('gain') for m in lgb_models])

importance_df = pd.DataFrame({
    'feature':         feature_cols,
    'mean_gain':       imp_per_fold.mean(axis=0),
    'std_gain':        imp_per_fold.std(axis=0),
    'min_gain':        imp_per_fold.min(axis=0),
    'n_folds_nonzero': (imp_per_fold > 0).sum(axis=0),
}).sort_values('mean_gain', ascending=False).reset_index(drop=True)

importance_df['stability'] = importance_df['n_folds_nonzero'] / CFG.N_FOLDS
# CV (coefficient of variation) = std/mean. Low = stable. High = noisy.
importance_df['cv'] = (importance_df['std_gain'] /
                       (importance_df['mean_gain'] + 1e-10)).round(3)

print('═══ TOP 30 MOST IMPORTANT FEATURES ═══')
cols_show = ['feature', 'mean_gain', 'stability', 'cv']
print(importance_df[cols_show].head(30).to_string(index=False))

n_zero     = (importance_df['mean_gain'] == 0).sum()
n_unstable = (importance_df['stability'] < 0.6).sum()
n_stable   = (importance_df['stability'] >= 0.8).sum()

print(f'\nFeature health summary:')
print(f'  Zero importance:         {n_zero}')
print(f'  Unstable (<0.6):         {n_unstable}  ← likely noise')
print(f'  Stable (≥0.8):           {n_stable}   ← reliable signal')

# Save for your investigation
importance_df.to_csv(CFG.OUTPUT_DIR / 'feature_importance.csv', index=False)
print('\nSaved → feature_importance.csv ✅')

═══ TOP 30 MOST IMPORTANT FEATURES ═══
                          feature  mean_gain  stability    cv
          S03_A02_A04_D03_F04_U02   6.066023        1.0 0.037
                      S02_F03_U01   5.510108        1.0 0.073
                      Price_LagT3   4.882185        1.0 0.097
          S03_A02_A04_D04_F04_U02   4.772356        1.0 0.117
    S03_A02_A04_D03_F04_U03_Acc23   4.241000        1.0 0.072
      S03_D02_A09_A02_B03_E03_E04   3.986434        1.0 0.087
                      S03_V04_T01   3.805013        1.0 0.222
      S03_D02_V01_A01_B10_E10_E11   3.769860        1.0 0.137
    S03_A02_A04_D03_F04_U03_Div13   3.604931        1.0 0.098
      S03_D02_A09_A02_B05_E05_E06   3.549108        1.0 0.169
                      S03_V05_T02   3.339874        1.0 0.170
                  S03_A07_A05_V17   3.145201        1.0 0.141
                  S03_D01_V12_D06   3.130199        1.0 0.107
                  S03_P01_D02_S03   3.086809        1.0 0.140
                S02_F01_U01_Lag

In [43]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — XGBOOST: 5-FOLD CV                               ║
# ╚══════════════════════════════════════════════════════════════╝
oof_xgb    = np.zeros(len(y), dtype=np.float64)
pred_xgb   = np.zeros(len(X_test), dtype=np.float64)
xgb_scores = []
xgb_iters  = []

# Clean param split: early_stopping_rounds in constructor only
xgb_ctor_params = {k: v for k, v in CFG.XGB_PARAMS.items()
                   if k != 'early_stopping_rounds'}
xgb_early_stop  = CFG.XGB_PARAMS['early_stopping_rounds']

print('══════════════════════════════════════════════')
print(f'XGBoost  —  {CFG.N_FOLDS}-Fold Cross-Validation')
print('══════════════════════════════════════════════')

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    t0 = time.time()
    X_trn, X_val = X[trn_idx], X[val_idx]
    y_trn, y_val = y[trn_idx], y[val_idx]

    model = xgb.XGBRegressor(
        **xgb_ctor_params,
        early_stopping_rounds=xgb_early_stop,
    )
    model.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        verbose=500,
    )

    val_preds = model.predict(X_val)
    oof_xgb[val_idx] = val_preds
    pred_xgb += model.predict(X_test) / CFG.N_FOLDS

    fold_r2 = r2_score(y_val, val_preds)
    xgb_scores.append(fold_r2)
    xgb_iters.append(model.best_iteration)
    print(f'  Fold {fold+1}/{CFG.N_FOLDS}  |  Val R²: {fold_r2:+.6f}  |  '
          f'iter: {model.best_iteration}  |  {time.time()-t0:.0f}s')
    del model; gc.collect()

xgb_oof_r2 = r2_score(y, oof_xgb)
xgb_std    = np.std(xgb_scores)

print(f'\n  XGB OOF R²:   {xgb_oof_r2:+.6f}')
print(f'  Fold std:     {xgb_std:.6f}')
print(f'  Stable?       {"✅ YES" if xgb_std < 0.002 else "⚠️  HIGH VARIANCE"}')
print(f'  Avg best iter:{np.mean(xgb_iters):.0f}')

══════════════════════════════════════════════
XGBoost  —  5-Fold Cross-Validation
══════════════════════════════════════════════
[0]	validation_0-rmse:0.02937
[500]	validation_0-rmse:0.02937
[1000]	validation_0-rmse:0.02936
[1040]	validation_0-rmse:0.02936
  Fold 1/5  |  Val R²: +0.000351  |  iter: 890  |  543s
[0]	validation_0-rmse:0.02954
[500]	validation_0-rmse:0.02953
[584]	validation_0-rmse:0.02953
  Fold 2/5  |  Val R²: +0.000280  |  iter: 434  |  302s
[0]	validation_0-rmse:0.02949
[500]	validation_0-rmse:0.02949
[1000]	validation_0-rmse:0.02949
[1316]	validation_0-rmse:0.02949
  Fold 3/5  |  Val R²: +0.000401  |  iter: 1166  |  695s
[0]	validation_0-rmse:0.02962
[500]	validation_0-rmse:0.02962
[674]	validation_0-rmse:0.02962
  Fold 4/5  |  Val R²: +0.000281  |  iter: 524  |  364s
[0]	validation_0-rmse:0.02954
[500]	validation_0-rmse:0.02954
[605]	validation_0-rmse:0.02954
  Fold 5/5  |  Val R²: +0.000335  |  iter: 455  |  321s

  XGB OOF R²:   +0.000335
  Fold std:     0.000045

In [44]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — CATBOOST: 5-FOLD CV (if available)               ║
# ╚══════════════════════════════════════════════════════════════╝
oof_cat    = np.zeros(len(y), dtype=np.float64)
pred_cat   = np.zeros(len(X_test), dtype=np.float64)
cat_scores = []
CATBOOST_DONE = False

if CATBOOST_AVAILABLE:
    print('══════════════════════════════════════════════')
    print(f'CatBoost  —  {CFG.N_FOLDS}-Fold Cross-Validation')
    print('══════════════════════════════════════════════')
    for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
        t0 = time.time()
        pool_trn = Pool(X[trn_idx], y[trn_idx])
        pool_val = Pool(X[val_idx], y[val_idx])
        model = CatBoostRegressor(**CFG.CAT_PARAMS)
        model.fit(pool_trn, eval_set=pool_val, use_best_model=True)
        val_preds = model.predict(X[val_idx])
        oof_cat[val_idx] = val_preds
        pred_cat += model.predict(X_test) / CFG.N_FOLDS
        fold_r2 = r2_score(y[val_idx], val_preds)
        cat_scores.append(fold_r2)
        print(f'  Fold {fold+1}/{CFG.N_FOLDS}  |  Val R²: {fold_r2:+.6f}  |  {time.time()-t0:.0f}s')
        del model, pool_trn, pool_val; gc.collect()
    cat_oof_r2 = r2_score(y, oof_cat)
    cat_std    = np.std(cat_scores)
    print(f'\n  CAT OOF R²:   {cat_oof_r2:+.6f}')
    print(f'  Fold std:     {cat_std:.6f}')
    CATBOOST_DONE = True
else:
    print('CatBoost not installed — skipping')

══════════════════════════════════════════════
CatBoost  —  5-Fold Cross-Validation
══════════════════════════════════════════════
0:	learn: 0.0295486	test: 0.0293693	best: 0.0293693 (0)	total: 495ms	remaining: 41m 16s
500:	learn: 0.0293567	test: 0.0292342	best: 0.0292342 (500)	total: 1m 52s	remaining: 16m 46s
1000:	learn: 0.0291952	test: 0.0291322	best: 0.0291322 (1000)	total: 3m 35s	remaining: 14m 21s
1500:	learn: 0.0290460	test: 0.0290417	best: 0.0290417 (1500)	total: 5m 23s	remaining: 12m 35s
2000:	learn: 0.0289099	test: 0.0289566	best: 0.0289566 (2000)	total: 7m 14s	remaining: 10m 51s
2500:	learn: 0.0287853	test: 0.0288819	best: 0.0288819 (2500)	total: 9m 1s	remaining: 9m 1s
3000:	learn: 0.0286651	test: 0.0288075	best: 0.0288075 (3000)	total: 10m 52s	remaining: 7m 14s
3500:	learn: 0.0285505	test: 0.0287396	best: 0.0287396 (3500)	total: 12m 35s	remaining: 5m 23s
4000:	learn: 0.0284417	test: 0.0286751	best: 0.0286751 (4000)	total: 14m 18s	remaining: 3m 34s
4500:	learn: 0.0283357	tes

In [45]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — ENSEMBLE STRATEGY (ONE-SHOT AWARE)               ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ONE-SHOT RULE:
# We PRINT both simple average and optimized weights.
# We AUTOMATICALLY choose the safer option:
#   - If optimized gain > 0.0005 AND fold std is low → use optimized
#   - Otherwise → use simple average
# Reasoning: with R²≈0.008, a 0.0003 gain from optimization is
# pure noise. Simple average is more robust to private LB shift.

oofs_list   = [oof_lgb, oof_xgb]
preds_list  = [pred_lgb, pred_xgb]
model_names = ['LGB', 'XGB']
stds_list   = [np.std(lgb_scores), np.std(xgb_scores)]

if CATBOOST_DONE:
    oofs_list.append(oof_cat)
    preds_list.append(pred_cat)
    model_names.append('CAT')
    stds_list.append(np.std(cat_scores))

print('══════════════════════════════════════════════')
print('ENSEMBLE — R² REPORT (your offline scorecard)')
print('══════════════════════════════════════════════')
for name, oof, std in zip(model_names, oofs_list, stds_list):
    r2 = r2_score(y, oof)
    print(f'  {name:<5} OOF R²: {r2:+.6f}  (fold std: {std:.6f})')

# Simple average
pred_sa  = np.mean(preds_list, axis=0)
oof_sa   = np.mean(oofs_list, axis=0)
r2_sa    = r2_score(y, oof_sa)
print(f'\n  Simple avg OOF R²:   {r2_sa:+.6f}')

# Optimized weights
def neg_r2(w):
    return -r2_score(y, sum(wi*o for wi, o in zip(w, oofs_list)))

n_m = len(oofs_list)
res = minimize(
    neg_r2,
    x0=[1/n_m]*n_m,
    method='SLSQP',
    bounds=[(0,1)]*n_m,
    constraints=[{'type':'eq', 'fun': lambda w: np.sum(w)-1}],
    options={'maxiter':500, 'ftol':1e-9}
)
opt_w   = np.clip(res.x, 0, 1);  opt_w /= opt_w.sum()
oof_opt = sum(w*o for w, o in zip(opt_w, oofs_list))
r2_opt  = r2_score(y, oof_opt)
gain    = r2_opt - r2_sa

print(f'  Optimized OOF R²:    {r2_opt:+.6f}  (gain: {gain:+.6f})')
print(f'  Optimal weights:     {dict(zip(model_names, opt_w.round(4)))}')

# Decision logic
max_std = max(stds_list)
print(f'\n  Max fold std:        {max_std:.6f}')
use_optimized = (gain > 0.0005) and (max_std < 0.003)

if use_optimized:
    pred_final = sum(w*p for w, p in zip(opt_w, preds_list))
    final_r2   = r2_opt
    method_str = 'Optimized weights'
    print('  → USING: Optimized weights (gain is meaningful)')
else:
    pred_final = pred_sa
    final_r2   = r2_sa
    method_str = 'Simple average'
    print('  → USING: Simple average (safer for one-shot private LB)')

print(f'\n  ★ FINAL OOF R²:      {final_r2:+.6f}  [{method_str}]')
print(f'  Pred mean:           {pred_final.mean():.6f}')
print(f'  Pred std:            {pred_final.std():.6f}')

══════════════════════════════════════════════
ENSEMBLE — R² REPORT (your offline scorecard)
══════════════════════════════════════════════
  LGB   OOF R²: +0.029897  (fold std: 0.000421)
  XGB   OOF R²: +0.000335  (fold std: 0.000045)
  CAT   OOF R²: +0.055276  (fold std: 0.000339)

  Simple avg OOF R²:   +0.031738
  Optimized OOF R²:    +0.055276  (gain: +0.023538)
  Optimal weights:     {'LGB': np.float64(0.0), 'XGB': np.float64(0.0), 'CAT': np.float64(1.0)}

  Max fold std:        0.000421
  → USING: Optimized weights (gain is meaningful)

  ★ FINAL OOF R²:      +0.055276  [Optimized weights]
  Pred mean:           0.000856
  Pred std:            0.003623


In [46]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — GENERATE submission.csv                          ║
# ║  *** THIS IS THE FILE YOU SUBMIT TO KAGGLE ***              ║
# ╚══════════════════════════════════════════════════════════════╝
print('Building submission...')

submission = pd.DataFrame({
    'ID':     test_ids,
    'TARGET': pred_final.astype(np.float64),
})

# ═══ 5 HARD ASSERTIONS — CRASH LOUD IF ANYTHING IS WRONG ═══
# In a one-shot competition, a silent bad submission = game over.
# These assertions ensure the file is 100% correct before saving.

assert len(submission) == len(test_ids), \
    f'FATAL: Length mismatch — {len(submission)} vs {len(test_ids)}'

assert submission['ID'].nunique() == len(submission), \
    'FATAL: Duplicate IDs in submission!'

assert submission['TARGET'].isnull().sum() == 0, \
    'FATAL: NaN values in TARGET predictions!'

assert not np.any(np.isinf(submission['TARGET'].values)), \
    'FATAL: Inf values in TARGET predictions!'

sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
assert set(submission['ID'].values) == set(sample_sub['ID'].values), \
    'FATAL: ID set does not match sample_submission.csv!'

# Sort by ID to match sample_submission order
submission = submission.sort_values('ID').reset_index(drop=True)

# Verify column names exactly match required format
assert list(submission.columns) == ['ID', 'TARGET'], \
    f'FATAL: Wrong column names: {list(submission.columns)}'

# Save
sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

# Verify file was actually written and is readable
verify = pd.read_csv(sub_path)
assert len(verify) == len(test_ids), 'FATAL: Saved file has wrong row count!'
assert list(verify.columns) == ['ID', 'TARGET'], 'FATAL: Saved file has wrong columns!'

print(f'\n✅ submission.csv SAVED AND VERIFIED')
print(f'   Path:     {sub_path}')
print(f'   Rows:     {len(submission):,}')
print(f'   Columns:  {list(submission.columns)}')
print(f'   ID range: {submission["ID"].min()} – {submission["ID"].max()}')
print()
print('First 10 rows:')
print(submission.head(10).to_string(index=False))
print()
print('TARGET distribution:')
print(submission['TARGET'].describe())

Building submission...

✅ submission.csv SAVED AND VERIFIED
   Path:     output\submission.csv
   Rows:     410,139
   Columns:  ['ID', 'TARGET']
   ID range: 672374 – 1089244

First 10 rows:
    ID    TARGET
672374  0.001265
672375 -0.013406
672376 -0.002619
672377  0.001281
672378  0.004916
672379 -0.005292
672380  0.000005
672381  0.002732
672382  0.000525
672383 -0.000106

TARGET distribution:
count    410139.000000
mean          0.000856
std           0.003623
min          -0.040947
25%          -0.001101
50%           0.000509
75%           0.002550
max           0.054782
Name: TARGET, dtype: float64


In [47]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — FINAL R² SCORECARD (self-investigation printout) ║
# ╚══════════════════════════════════════════════════════════════╝
#
# This is your COMPLETE offline estimate of how you will perform.
# OOF R² on 660k training rows is more reliable than public LB
# (which is only 26% of 410k test rows = ~107k rows).

print()
print('╔' + '═'*63 + '╗')
print('║   R² SCORECARD — iRage Short-Horizon Return Prediction   ║')
print('╠' + '═'*63 + '╣')
print(f'║  Features (original):  {445:<39d} ║')
print(f'║  Features (after FE):  {n_features:<39d} ║')
print(f'║  Train rows:           {len(X):<39,} ║')
print(f'║  Test rows:            {len(X_test):<39,} ║')
print('╠' + '═'*63 + '╣')
print('║  INDIVIDUAL MODEL OOF R²                                 ║')
for name, oof in zip(model_names, oofs_list):
    r2 = r2_score(y, oof)
    bar = '█' * max(0, int(r2 * 5000))  # visual bar
    print(f'║  {name:<5}  {r2:+.6f}  {bar:<40} ║')
print('╠' + '═'*63 + '╣')
print('║  ENSEMBLE                                                ║')
print(f'║  Simple avg:    {r2_sa:+.6f}                                ║')
print(f'║  Optimized:     {r2_opt:+.6f}                                ║')
print(f'║  ★ FINAL ({method_str:<16}): {final_r2:+.6f}               ║')
print('╠' + '═'*63 + '╣')
print('║  OVERFIT CHECK                                           ║')
print(f'║  Train R²: {train_r2_diag:.6f}  |  Val R²: {val_r2_diag:.6f}           ║')
ovf_str = f'Gap: {train_r2_diag-val_r2_diag:.6f} — {"✅ LOW" if (train_r2_diag-val_r2_diag)<0.01 else "⚠️  CHECK"}'
print(f'║  {ovf_str:<62}║')
print('╠' + '═'*63 + '╣')
print('║  INTERPRETATION (quant context)                          ║')
if final_r2 > 0.005:
    level = 'STRONG ★ — ***___***'
elif final_r2 > 0.003:
    level = 'GOOD  — competitive, top-10 range'
elif final_r2 > 0.001:
    level = 'DECENT — above majority of field'
elif final_r2 > 0:
    level = 'WEAK  — positive signal, but small'
else:
    level = 'NO SIGNAL — model is predicting noise'
print(f'║  {level:<62}║')
print('╠' + '═'*63 + '╣')
print('║  OUTPUT FILES                                            ║')
print('║  /kaggle/working/submission.csv        ← SUBMIT THIS    ║')
print('║  /kaggle/working/feature_importance.csv ← investigate   ║')
print('╚' + '═'*63 + '╝')
print()
print('✅ NOTEBOOK COMPLETE')
print('   Run top-to-bottom = submission.csv + full R² scorecard')


╔═══════════════════════════════════════════════════════════════╗
║   R² SCORECARD — iRage Short-Horizon Return Prediction   ║
╠═══════════════════════════════════════════════════════════════╣
║  Features (original):  445                                     ║
║  Features (after FE):  796                                     ║
║  Train rows:           661,574                                 ║
║  Test rows:            410,139                                 ║
╠═══════════════════════════════════════════════════════════════╣
║  INDIVIDUAL MODEL OOF R²                                 ║
║  LGB    +0.029897  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ ║
║  XGB    +0.000335  █                                        ║
║  CAT    +0.055276  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

---
## 📌 ONE-SHOT SUBMISSION CHECKLIST

Before you click **Submit Prediction** on Kaggle, verify:

- [ ] **Cell 8 gap < 0.01** — overfit check passed
- [ ] **Final OOF R² > 0** — model has positive signal
- [ ] **Fold std < 0.003** — predictions are stable across folds
- [ ] **Cell 14 printed `✅ submission.csv SAVED AND VERIFIED`**
- [ ] **submission.csv has 410,139 rows** (same as test set)
- [ ] **Two columns only: ID, TARGET**
- [ ] **No NaN, no Inf in TARGET column**

If ALL boxes checked → safe to submit.

---
## 📌 What the R² scorecard means in context

| Your OOF R² | Estimated rank | Notes |
|---|---|---|
| > 0.005 | Top 3 | Above current leaderboard top |
| 0.003–0.005 | Top 5–10 | Competitive, prize range |
| 0.001–0.003 | Top 15–30 | Above majority |
| 0–0.001 | Top 30–60 | Weak positive signal |
| < 0 | Bottom half | Model predicting noise |

Remember: private LB = **74% of test data**. Even with OOF R²≈0.003, your submission can beat people who overfit to the public 26%.